# Language Models are Few-Shot Learners (GPT-3) — Implementation / 구현

**Paper**: Brown, T. B. et al. "Language Models are Few-Shot Learners". NeurIPS 2020. arXiv:2005.14165

**한국어** — 이 노트북은 GPT-3 본문에서 가장 의미 있는 발견 — *gradient update 없이 prompt 안의 demonstration만으로 새 작업을 수행하는 in-context learning* — 을 작은 사전학습 GPT-2 모델로 재현합니다. 175B 모델은 우리가 직접 돌릴 수 없지만, 같은 architecture 가족(decoder-only autoregressive Transformer)의 작은 모델로 **zero-shot vs one-shot vs few-shot** 차이가 실제로 존재함을 보일 수 있습니다.

**English** — This notebook reproduces GPT-3's central finding — *in-context learning, where new tasks are performed purely from prompt demonstrations with no gradient updates* — using a small pretrained GPT-2 model. We cannot run 175B parameters locally, but the same architectural family (decoder-only autoregressive Transformer) lets us demonstrate the **zero- vs one- vs few-shot** gap on toy tasks.

**Toy tasks / 토이 작업**:
1. Single-digit addition (e.g. "5 + 7 = ?") — replicates §3.9.1 / 한 자리 덧셈, §3.9.1 재현
2. Sentiment classification (positive / negative) — task-description vs demonstrations / 감정 분류, 지시 vs 예시 비교
3. In-context learning curve: accuracy vs $K$ (replicates Figure 1.2) / $K$에 따른 정확도 곡선, Figure 1.2 재현

In [ ]:
import re
import random
from typing import List, Tuple

import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

## Part 1: Load a Pretrained Decoder-Only Transformer / 사전학습 Decoder-Only Transformer 로드

**한국어** — GPT-3 175B는 96 layer, 12288 d_model, 96 head이지만 우리는 GPT-2 small(124M, 12 layer, 768 d_model)을 사용합니다. 이 둘은 architecturally 동일하며, GPT-3 paper의 Table 2.1에서 GPT-3 Small (125M, 12L, 768d, 12H)과 거의 일치하는 사이즈입니다. 즉 우리는 **GPT-3 family의 가장 작은 모델과 같은 스케일**로 in-context learning을 시연합니다.

**English** — GPT-3 175B has 96 layers, $d_\text{model}=12288$, 96 heads. We use GPT-2 small (124M, 12L, 768d) — architecturally identical and almost exactly the size of GPT-3 Small (Table 2.1: 125M, 12L, 768d, 12H). So we are running ICL at the smallest GPT-3 scale.

**Important — no fine-tuning, no training, only forward passes / 학습 없음, forward pass만**:

In [ ]:
MODEL_NAME = 'gpt2'  # 124M params, smallest GPT-2 ~ matches GPT-3 Small
tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()  # frozen — never updated

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {n_params/1e6:.1f}M  (compare GPT-3 Small 125M, GPT-3 175B = 175,000M)")
print(f"Layers: {model.config.n_layer}, d_model: {model.config.n_embd}, heads: {model.config.n_head}")
print(f"Context window: {model.config.n_ctx}")

## Part 2: The In-Context Learning Function / In-Context Learning 함수

**한국어** — GPT-3의 핵심 동작을 단 한 함수로 캡슐화. `task_description`과 `K`개의 `(input, output)` 예시를 받아 prompt를 만들고, frozen 모델로부터 다음 토큰들을 greedy decoding합니다. **gradient도, parameter update도 없음** — 모든 것이 forward pass 안에서 일어남.

**English** — Capsules GPT-3's core inference behavior in a single function. It builds a prompt from the task description, $K$ demonstrations, and the test input, then greedy-decodes from the frozen model. **No gradients, no parameter updates** — everything happens within forward passes.

In [ ]:
def build_prompt(task_description: str,
                 demonstrations: List[Tuple[str, str]],
                 test_input: str,
                 input_prefix: str = 'Q: ',
                 output_prefix: str = 'A: ') -> str:
    """Construct an in-context learning prompt (GPT-3 style).

    Args:
        task_description: Natural-language instruction (zero-shot can use this alone).
        demonstrations: List of (input, output) tuples, K of them.
        test_input: The input to predict on.
        input_prefix: String marker before each input.
        output_prefix: String marker before each output.

    Returns:
        Prompt string ready to be tokenized.
    """
    parts = []
    if task_description:
        parts.append(task_description)
    for x, y in demonstrations:
        parts.append(f"{input_prefix}{x}\n{output_prefix}{y}")
    parts.append(f"{input_prefix}{test_input}\n{output_prefix}")
    return "\n\n".join(parts)


@torch.no_grad()
def in_context_predict(prompt: str,
                       max_new_tokens: int = 8,
                       stop_at_newline: bool = True) -> str:
    """Greedy-decode a continuation from a frozen LM. No gradient, no update.

    Args:
        prompt: The full GPT-3-style prompt.
        max_new_tokens: Maximum tokens to generate after the prompt.
        stop_at_newline: If True, truncate the answer at the first newline.

    Returns:
        The generated continuation string (whitespace-stripped).
    """
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # greedy
        pad_token_id=tokenizer.eos_token_id,
    )
    completion = tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    if stop_at_newline:
        completion = completion.split('\n')[0]
    return completion.strip()


# Quick sanity check
demo_prompt = build_prompt(
    task_description="Translate English to French:",
    demonstrations=[("sea otter", "loutre de mer"),
                    ("plush giraffe", "girafe peluche")],
    test_input="cheese",
)
print("=== Prompt ===")
print(demo_prompt)
print("\n=== Continuation ===")
print(in_context_predict(demo_prompt))

## Part 3: Single-Digit Arithmetic — §3.9.1 Replication / 한자리 산술 재현

**한국어** — GPT-3 paper Figure 3.10에서 175B는 2-digit 덧셈 100%, 13B는 ~10%로 phase transition을 보였습니다. 우리는 GPT-2 small에서 **단자리** 덧셈으로 동일한 setup의 ICL 효과만 확인 — zero-shot은 구조를 모르고, few-shot은 패턴을 픽업.

**English** — GPT-3 paper's Figure 3.10 shows a phase transition: 175B at ~100% on 2-digit addition while 13B at ~10%. With GPT-2 small we shrink to single-digit addition, just to demonstrate the ICL setup — zero-shot doesn't know the format, few-shot picks up the pattern.

In [ ]:
def make_addition_problems(n: int, max_digit: int = 9, seed: int = 1) -> List[Tuple[str, str]]:
    """Generate single-digit addition problems formatted as (question, answer).

    Args:
        n: Number of problems.
        max_digit: Inclusive upper bound for each operand.
        seed: RNG seed.

    Returns:
        List of (question, answer-string) tuples.
    """
    rng = random.Random(seed)
    problems = []
    for _ in range(n):
        a = rng.randint(0, max_digit)
        b = rng.randint(0, max_digit)
        problems.append((f"What is {a} plus {b}?", str(a + b)))
    return problems


def extract_first_int(text: str) -> str:
    """Pull the leading integer from a model continuation, or return empty string.

    Args:
        text: Free-form continuation.

    Returns:
        The first integer found, as a string.
    """
    m = re.search(r"-?\d+", text)
    return m.group(0) if m else ""


def evaluate_arithmetic(K: int, n_eval: int = 30, seed: int = 42) -> float:
    """Evaluate K-shot accuracy on single-digit addition.

    Args:
        K: Number of in-context demonstrations.
        n_eval: Number of evaluation problems.
        seed: RNG seed for evaluation set.

    Returns:
        Accuracy in [0, 1].
    """
    eval_set = make_addition_problems(n_eval, seed=seed)
    demo_pool = make_addition_problems(50, seed=seed + 1000)
    correct = 0
    for q, gold in eval_set:
        rng = random.Random(seed + hash(q) % 10000)
        demos = rng.sample(demo_pool, K) if K > 0 else []
        prompt = build_prompt(
            task_description="Add the two numbers.",
            demonstrations=demos,
            test_input=q,
        )
        ans = extract_first_int(in_context_predict(prompt, max_new_tokens=4))
        if ans == gold:
            correct += 1
    return correct / n_eval


# Show one full prompt + answer trace for K=4 (mirrors Figure 1.1 of paper)
trace_demos = make_addition_problems(4, seed=7)
trace_prompt = build_prompt("Add the two numbers.", trace_demos, "What is 6 plus 3?")
print("=== K=4 Few-Shot Prompt ===\n")
print(trace_prompt)
print("\n=== Model Continuation ===")
print(in_context_predict(trace_prompt, max_new_tokens=4))

## Part 4: In-Context Learning Curve / In-Context Learning 곡선

**한국어** — GPT-3 paper Figure 1.2의 핵심 plot 재현 — accuracy vs number of in-context examples ($K$). 이론적으로 ICL이 작동하면 $K$가 늘어날수록 정확도가 증가해야 합니다. 작은 모델에서는 기울기가 완만, 큰 모델에서는 가파릅니다 (paper의 발견).

**English** — Reproduces the core plot of GPT-3's Figure 1.2: accuracy as a function of number of in-context demonstrations ($K$). If ICL works, the curve goes up with $K$. The paper's main finding was that **larger models have steeper curves** — let us see what GPT-2 124M's looks like.

In [ ]:
K_values = [0, 1, 2, 4, 8, 16]
accs = [evaluate_arithmetic(K=K, n_eval=20) for K in K_values]

for K, acc in zip(K_values, accs):
    print(f"K={K:2d}  →  accuracy = {acc:.2f}")

fig, ax = plt.subplots()
ax.plot(K_values, accs, marker='o', linewidth=2, markersize=10, color='tab:blue', label='GPT-2 124M')
ax.set_xlabel('Number of in-context examples (K) / In-context 예시 수')
ax.set_ylabel('Accuracy / 정확도')
ax.set_title('In-Context Learning Curve on Single-Digit Addition\n'
             '단자리 덧셈에서의 In-Context Learning 곡선 (cf. GPT-3 paper Fig 1.2)')
ax.set_ylim(-0.05, 1.05)
ax.axhline(0.1, color='gray', linestyle='--', alpha=0.5, label='~chance for 0-19 range')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Part 5: Sentiment Classification — Zero/One/Few-Shot / 감정 분류 — Zero/One/Few-shot

**한국어** — 두 번째 토이 작업: 영화 리뷰 감정 분류. GPT-3 paper §2.4 (Evaluation)에서 binary classification은 "True"/"False" 같은 의미 있는 라벨로 reformulate하는 것이 좋다고 함. 우리는 "positive"/"negative"를 사용. 동일한 모델, 동일한 task, K만 바꿔서 prompt를 비교.

**English** — Second toy task: movie-review sentiment. GPT-3 paper §2.4 recommends recasting binary tasks with meaningful labels ("True"/"False" or here "positive"/"negative"). Same model, same task, only $K$ changes.

In [ ]:
DEMO_POOL = [
    ("This movie was absolutely fantastic, I loved every minute.", "positive"),
    ("I have never been so bored in my life. Total waste of time.", "negative"),
    ("A delightful and heartwarming story, beautifully acted.", "positive"),
    ("The plot was incoherent and the acting was wooden.", "negative"),
    ("Brilliant cinematography and a gripping screenplay.", "positive"),
    ("Painfully slow, with characters I didn't care about.", "negative"),
]

EVAL_SET = [
    ("An incredible journey full of memorable moments.", "positive"),
    ("Tedious, predictable, and far too long.", "negative"),
    ("Genuinely funny and surprisingly touching.", "positive"),
    ("Confusing storyline and bland visuals.", "negative"),
    ("A masterpiece that will be remembered for decades.", "positive"),
    ("I almost walked out halfway through.", "negative"),
    ("Unforgettable performances by the entire cast.", "positive"),
    ("Disappointing on every level.", "negative"),
]

TASK_DESC = ('Classify the sentiment of the review as either "positive" or "negative".')


def evaluate_sentiment(K: int, seed: int = 0) -> Tuple[float, List[Tuple[str, str, str]]]:
    """Evaluate K-shot accuracy on toy sentiment classification.

    Args:
        K: Number of in-context demonstrations (sampled from DEMO_POOL).
        seed: RNG seed for demo selection.

    Returns:
        (accuracy, per_example_records) where each record is (review, gold, pred).
    """
    rng = random.Random(seed)
    records = []
    correct = 0
    for review, gold in EVAL_SET:
        demos = rng.sample(DEMO_POOL, K) if K > 0 else []
        prompt = build_prompt(
            task_description=TASK_DESC,
            demonstrations=demos,
            test_input=review,
            input_prefix='Review: ',
            output_prefix='Sentiment: ',
        )
        out = in_context_predict(prompt, max_new_tokens=3).lower()
        # First word match
        pred = 'positive' if 'positive' in out else ('negative' if 'negative' in out else out.split()[0] if out else '')
        records.append((review, gold, pred))
        if pred == gold:
            correct += 1
    return correct / len(EVAL_SET), records


results = {}
for K in [0, 1, 2, 4, 6]:
    acc, recs = evaluate_sentiment(K)
    results[K] = acc
    print(f"--- K = {K} ---  accuracy = {acc:.2f}")
    for r in recs[:3]:
        print(f"   review='{r[0][:50]}...'  gold={r[1]}  pred={r[2]}")

fig, ax = plt.subplots()
Ks = sorted(results.keys())
ax.plot(Ks, [results[k] for k in Ks], marker='s', linewidth=2, markersize=10, color='tab:orange')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='random chance / 무작위 추측')
ax.set_xlabel('Number of in-context examples (K) / In-context 예시 수')
ax.set_ylabel('Accuracy / 정확도')
ax.set_title('Sentiment Classification: Zero / One / Few-Shot\n'
             '감정 분류: Zero / One / Few-Shot')
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Part 6: Summary — What We Reproduced and What We Didn't / 요약 — 재현한 것과 못한 것

**한국어**

| 개념 / Concept | This Notebook (GPT-2 124M) | GPT-3 Paper (175B) |
|---|---|---|
| Architecture | Decoder-only Transformer | Decoder-only Transformer (alternating dense/sparse) |
| Parameters | 124M (12L, 768d) | 175B (96L, 12288d) |
| Context window | 1024 tokens | 2048 tokens |
| In-context learning paradigm | ✅ Demonstrated | ✅ Demonstrated and quantified |
| Zero/One/Few-shot prompts | ✅ Built and evaluated | ✅ Standard evaluation protocol |
| No gradient updates at test time | ✅ `model.eval()`, `torch.no_grad` | ✅ Frozen 175B weights |
| Phase transition with scale | ❌ Not visible at 124M | ✅ 13B → 175B jump on arithmetic |
| LAMBADA 86.4%, TriviaQA 71.2% | ❌ Need 175B | ✅ Headline numbers |
| Human-level news article generation | ❌ Need 175B | ✅ 52% detection (chance) |

**핵심 메시지** — GPT-3의 ICL paradigm은 **architecture 변경 없이** 작동합니다. 같은 transformer, 같은 자기회귀 next-token prediction loss, 같은 prompting 형식 — 단지 파라미터를 1000× 키우면 능력이 따라옵니다. 우리 124M 모델에서도 ICL의 *기능*은 보이지만 (sentiment에서 K↑→accuracy↑), GPT-3 paper의 *quantitative magnitude*에 도달하려면 scale이 필요합니다.

**English**

**Key message** — GPT-3's ICL paradigm works **without any architectural change**. The same transformer, the same autoregressive next-token loss, the same prompting format — scale parameters 1000× and capability follows. Even our 124M model shows the *mechanism* of ICL (accuracy rises with $K$ on sentiment), but reaching the *quantitative magnitude* in the paper (LAMBADA 86%, arithmetic 100%) requires scale.

**Connection to other papers / 다른 논문과의 연결** — Kaplan et al. (#39, scaling laws) provided the theoretical extrapolation that justified going to 175B; Chinchilla (2022) later showed GPT-3 was data-undertrained; InstructGPT (2022) added RLHF on top of this exact base. This notebook's prompting interface is also exactly what ChatGPT, Claude, GPT-4, and every modern LLM expose.